# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.


## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Мои предсказания и объяснения:

1. 
2. 
3. 
4. 
5. 

In [17]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [18]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

5463 строк


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [19]:
# База
def to_num(s):
    s = s.replace(" ", "")
    return float(s)
    


In [20]:
CITY_MAP = {
    "almaty": "Алматы",
    "astana": "Астана",
    "shymkent": "Шымкент",
}

def clean_city(s):
    cleaned = s.strip().lower()
    return CITY_MAP.get(cleaned, cleaned.title())

{clean_city(r["city"]) for r in rows}

{'Актобе', 'Алматы', 'Астана', 'Атырау', 'Караганда', 'Шымкент'}

In [ ]:

cat_count = {}
for r in rows:
    cat = r["category"]
    cat_count[cat] = cat_count.get(cat, 0) + 1

for cat, n in sorted(cat_count.items(), key=lambda p: p[1], reverse=True):
    print(f"{cat:<15}{n:>6}")

In [ ]:
revenue = {}   
count = {}     

for r in rows:
    if r["status"] == "оплачен":
        cat = r["category"]
        revenue[cat] = revenue.get(cat, 0) + to_num(r["amount"])
        count[cat] = count.get(cat, 0) + 1

print(f"{'Категория':<15}{'Выручка':>15}{'Средний чек':>15}")
for cat in sorted(revenue, key=revenue.get, reverse=True):
    avg = revenue[cat] / count[cat]
    print(f"{cat:<15}{revenue[cat]:>15,.0f}{avg:>15,.0f}")

Категория              Выручка    Средний чек
Электроника         65,270,240         61,287
Одежда              18,321,080         13,286
Дом                 10,516,320         10,655
Красота              5,507,620          6,188
Книги                2,294,100          3,935


In [ ]:
# 4
by_month = {}
for r in rows:
    month = r["order_date"][:7]
    by_month[month] = by_month.get(month, 0) + 1

for month in sorted(by_month):
    print(month, by_month[month])

## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [ ]:
# Стандарт
# 5
rev_month = {}
for r in rows:
    if r["status"] == "оплачен":
        m = r["order_date"][:7]
        rev_month[m] = rev_month.get(m, 0) + to_num(r["amount"])

for m in sorted(rev_month):
    print(f"{m}: {rev_month[m]:,.0f}")

sep = rev_month["2026-09"]
oct_ = rev_month["2026-10"]
print(f"Октябрь к сентябрю: {(oct_ - sep) / sep * 100:+.1f}%")

In [ ]:
# 6
total = {}
cancelled = {}
for r in rows:
    city = clean_city(r["city"])
    total[city] = total.get(city, 0) + 1
    if r["status"] == "отменён":
        cancelled[city] = cancelled.get(city, 0) + 1

for city in sorted(total):
    share = cancelled.get(city, 0) / total[city] * 100
    print(f"{city:<12}{share:>6.1f}%")

In [ ]:
# 7. Топ-5 клиентов по выручке оплаченных заказов
rev_cust = {}
for r in rows:
    if r["status"] == "оплачен":
        c = r["customer_id"]
        rev_cust[c] = rev_cust.get(c, 0) + to_num(r["amount"])

top5 = sorted(rev_cust.items(), key=lambda p: p[1], reverse=True)[:5]
for c, v in top5:
    print(f"{c}: {v:,.0f}")

# заказы лидера
for r in rows:
    if r["customer_id"] == top5[0][0]:
        print(r["order_date"], r["category"], r["amount"], r["status"])

**Ответ к задаче 7.** На первом месте клиент `TEST` — это не настоящий покупатель, а тестовые заказы. У него 3 заказа «Электроники» по 9 999 990 (12.09, 25.09 и 03.10), в сумме почти 30 млн — больше половины месячной выручки магазина. Такие строки нужно исключать из анализа, иначе выручка сильно завышена, особенно в сентябре (там два из трёх тестовых заказов).

In [ ]:
# 8.
cust_city = {}
for r in rows:
    city = clean_city(r["city"])
    if city not in cust_city:
        cust_city[city] = set()
    cust_city[city].add(r["customer_id"])

for city in sorted(cust_city):
    print(f"{city:<12}{len(cust_city[city]):>5}")

## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [23]:
# Челлендж